In [68]:
import os
import sys
import glob
import pickle
import random
import shutil
import zipfile
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as opt
from torch.utils.data import Dataset, DataLoader

### FILES ###
import shutil # delate directory
import zipfile # extract a directory

from sklearn.model_selection import train_test_split # train and test split

In [69]:
# DELETE A DIRECTORY FROM CONTENT

folder = "DATASET_SHARP\doppler_traces"

if os.path.exists(folder):
    shutil.rmtree(folder)
    print("Delated directory")
else:
    print("No delated directory")

Delated directory


In [70]:
# RETURN THE ACTIONS FOR EACH DIRECTORY

def getActions(folder_path):

    actions = []

    for filename in os.listdir(folder_path):

        x = filename.split("_")

        action = x[1]
        if len(action) > 1:
            action =  action[0]

        if action not in actions:
            actions.append(action)

    actions.sort()

    print("Actions:", actions)

    return actions

In [71]:
import zipfile
import os

zip_path = os.path.join("DATASET_SHARP", "doppler_traces.zip")
extract_path = os.path.join("DATASET_SHARP", "doppler_traces")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Estrazione completata.")

Estrazione completata.


In [72]:
# LISTA DELLE CARTELLE DEL DATASET

ROOT_PATH = "DATASET_SHARP\doppler_traces"
#print(os.listdir(ROOT_PATH))
folders = []
for set in os.listdir(ROOT_PATH):

    sets_path = os.path.join(ROOT_PATH, set)

    for folder_name in sorted(os.listdir(sets_path)):

        folder_path = os.path.join(sets_path, folder_name)

        # Considera solo le cartelle
        if not os.path.isdir(folder_path):
            continue

        folders.append(folder_path)

print(folders)

['DATASET_SHARP\\doppler_traces\\doppler_traces\\S1a', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S1b', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S1c', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S2a', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S2b', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S3a', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S4a', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S4b', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S5a', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S6a', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S6b', 'DATASET_SHARP\\doppler_traces\\doppler_traces\\S7a']


In [73]:
# EXTRACT DATASET

complete_dataset = {}

for i in range(0, len(folders)):

    folder = folders[i]
    actions = getActions(folder)
    dataset_array = {}

    for action in actions:
        dataset_array[action] = []

    folder_name = os.path.basename(folder)
    #print(folder_name)
    for action in actions:
        #print(action)
        for filename in os.listdir(folder):
            #print(filename)
            marker = f"_{action}"
            if marker in filename:
                #print("aaa")
                file_path = os.path.join(folder, filename)

                with open(file_path, "rb") as fp:
                    doppler = pickle.load(fp)
                #print(doppler)
                #print(f"{action} add {doppler}")
                dataset_array[action].append(doppler)

    complete_dataset[folder_name] = dataset_array

Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']
Actions: ['E', 'J', 'L', 'R', 'W']
Actions: ['E', 'J', 'L', 'R', 'W']
Actions: ['E', 'H', 'J', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']
Actions: ['E', 'J', 'L', 'R', 'W']
Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'W']
Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']
Actions: ['E', 'J', 'L', 'R', 'W']
Actions: ['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']


In [74]:
# WINDOW OF 1@(340 x 100)

def create_sliding_windows(complete_dataset, window_length=340, stride=340):

  X = []
  y = []
  folders = []
  streams = []
 ############################################################
  #print(complete_dataset) ### J is empty ?????
  ######################################################
  for folder_name in complete_dataset:
    print("Cartella: ", folder_name)
    dataset = complete_dataset[folder_name]
    all_windows = {}

    for action in dataset:

      #data = np.asarray(dataset[action])
      data = [np.asarray(x) for x in dataset[action]]
      windows_activity = []
      # elements of each action
      #num_streams, time_length, num_features = np.array(data).shape
      #print(f"Action {action} -> Shape of data: {num_streams}, {time_length}, {num_features}")

      num_streams = len(data)
      #print(f"Action {action} -> num_streams: {num_streams}")

      for stream in range(num_streams):

        stream_data = data[stream]
        time_length, num_features = stream_data.shape
        window = []
        # se lo stream è più corto in partenza della grandezza della finestra
        if time_length < window_length:
          print("The dataset is less than window size")
          continue

        start = 0
        end = window_length
        while end <= time_length:
          window = data[stream][start:end,:]
          #windows_stream.append(window)
          X.append(window)
          y.append(action)
          folders.append(folder_name)
          streams.append(stream)
          start += stride
          end += stride

        #if start < time_length and end > time_length:
          #last_window = np.array(data[stream][start:time_length,:])
          #print("Zero da aggiungere: ", window_length - last_window.shape[0])
          #zero_padding = np.zeros((window_length - last_window.shape[0], last_window.shape[1]), dtype=last_window.dtype)
          #last_window = np.vstack((last_window, zero_padding))
          #windows_stream.append(last_window)
          #print("Shape ultima finestra: ", last_window.shape)
          #print("NUmbers of zeros: ", len(zero_padding))
          #print("last window: ", last_window)
      print(f"Action {action} -> Shape of data: {num_streams}, {time_length}, {num_features}")
      del data

    del dataset

  return X, y, folders, streams

X, y, folders, streams = create_sliding_windows(complete_dataset)

#print("\n==========================")
#print("DATASET FINALE")
#print("==========================")

#for folder_name in complete_dataset:

    #print(f"\nCartella: {folder_name}")

    #for action in complete_dataset[folder_name]:

        #print(
            #f"  {action}: {complete_dataset[folder_name][action].shape}"
        #)


Cartella:  S1a
Action C -> Shape of data: 4, 18766, 100
Action E -> Shape of data: 4, 18700, 100
Action H -> Shape of data: 4, 19064, 100
Action J -> Shape of data: 8, 8708, 100
Action L -> Shape of data: 4, 18842, 100
Action R -> Shape of data: 4, 19269, 100
Action S -> Shape of data: 4, 19009, 100
Action W -> Shape of data: 4, 19295, 100
Cartella:  S1b
Action C -> Shape of data: 4, 18803, 100
Action E -> Shape of data: 4, 18270, 100
Action H -> Shape of data: 4, 18707, 100
Action J -> Shape of data: 8, 8667, 100
Action L -> Shape of data: 4, 19329, 100
Action R -> Shape of data: 4, 19253, 100
Action S -> Shape of data: 4, 18922, 100
Action W -> Shape of data: 4, 18927, 100
Cartella:  S1c
Action C -> Shape of data: 4, 18533, 100
Action E -> Shape of data: 4, 18929, 100
Action H -> Shape of data: 4, 19053, 100
Action J -> Shape of data: 8, 8784, 100
Action L -> Shape of data: 4, 19547, 100
Action R -> Shape of data: 4, 19197, 100
Action S -> Shape of data: 4, 19017, 100
Action W -> Sha

In [75]:
index = np.random.randint(0, len(X))
print("Index: ", index)
print(type(X[index]))
print("Element in X: ", X[index].shape)
print("Element in y: ", y[index])
print("Element in folders: ", folders[index])
print("Element in streams: ", streams[index])

Index:  7023
<class 'numpy.ndarray'>
Element in X:  (340, 100)
Element in y:  E
Element in folders:  S2b
Element in streams:  0


In [76]:
label_map = {
    "W": 0,
    "R": 1,
    "J": 2,
    #"J2": 2,
    "L": 3,
    "S": 4,
    "C": 5,
    "G": 6,
    "E": 7,
    "H": 8
}

class DopplerDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        element = self.dataset[idx]
        # Convert to float32
        x = torch.from_numpy(element["data"]).float()
        # Add channel dimension -> (1, 340, 100)
        x = x.unsqueeze(0)

        activity = element["label"]

        y = torch.tensor(label_map[activity], dtype=torch.long)

        z = element["folder"]

        w = element["stream"]

        return x, y, z, w


In [77]:
#print(type(dataset[0]))

In [78]:
# DATASET, TRAINING, TEST
dataset = [
    {
        "data": data,
        "label": label,
        "folder": folder,
        "stream": stream
    }
    for data, label, folder, stream
    in zip(X, y, folders, streams)
]

dataset_S1 = []
dataset_test_external = []

for sample in dataset:
  if sample["folder"].startswith("S1"):
    dataset_S1.append(sample)
  else:
    dataset_test_external.append(sample)

labels = []

for sample in dataset_S1:
    labels.append(sample["label"])

unique_labels = sorted({s["label"] for s in dataset_S1})
print(unique_labels)

doppler_dataset = DopplerDataset(dataset_S1)
train_S1_dataset, test_S1_dataset = train_test_split(
    doppler_dataset,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

labels_train = []

for sample in train_S1_dataset:
    labels_train.append(sample[1])

['C', 'E', 'H', 'J', 'L', 'R', 'S', 'W']


In [79]:
# CONTENUTO CODICE

print("Dataset: ", dataset[0])
print(dataset[0]["data"].shape)
print("Dataset completo:", len(dataset))
print("Dataset S1:", len(dataset_S1))
print("Dataset test esterno:", len(dataset_test_external))
print("Train S1:", len(train_S1_dataset))
print("Test S1:", len(test_S1_dataset))

#print(train_S1_dataset[0]["data"].shape)
print(train_S1_dataset[0][0].shape)
from collections import Counter

#print(Counter(labels_train))

Dataset:  {'data': array([[0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       ...,
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573]], shape=(340, 100)), 'label': 'C', 'folder': 'S1a', 'stream': 0}
(340, 100)
Dataset completo: 18680
Dataset S1: 5240
Dataset test esterno: 13440
Train S1: 4192
Test S1: 1048
torch.Size([1, 340, 100])


In [80]:
#Network Definition

NUM_LABELS = 9

class BaseNet(nn.Module):
    def __init__(self):
        super().__init__()

        print("Network Initialized")

        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=5, kernel_size=2, stride=2),
            nn.ReLU())
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=3, kernel_size=1, stride=1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(in_channels=3, out_channels=6, kernel_size=2, stride=1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(in_channels=6, out_channels=9, kernel_size=4, stride=2, padding=1),
            nn.ReLU()
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=15, out_channels=3, kernel_size=1, stride=1, padding='same'),
            nn.ReLU()
        )

        self.block2 = nn.Sequential(
            nn.Dropout(0.2),
            nn.LazyLinear(out_features=NUM_LABELS)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        print("Branch 1:", b1.shape)
        b2 = self.branch2(x)
        print("Branch 2:", b2.shape)
        b3 = self.branch3(x)
        print("Branch 3:", b3.shape)
        h = torch.cat([b1, b2, b3], dim=1)
        print("Concat:", h.shape)
        h = self.block1(h)
        print("Concat:", h.shape)
        h = h.flatten(1)
        print("Flatten:", h.shape)
        out = self.block2(h)
        print("Out:", out.shape)

        return out


In [81]:
batch_size = 64
num_workers = 0
pin_memory = torch.cuda.is_available()

train_dataloader = DataLoader(train_S1_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=pin_memory)
#valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False,
#                              num_workers=num_workers, pin_memory=pin_memory)
test_dataloader = DataLoader(test_S1_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=pin_memory)

In [82]:
model = BaseNet()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = opt.Adam(model.parameters(), lr=0.0001)

loss_fn = nn.CrossEntropyLoss()

epochs = 1

for epoch in range(epochs):
    model.train()
    print(f"Epoch:{epoch+1}")
    train_iterator = tqdm(train_dataloader)
    for batch_x, batch_y, _, _ in train_iterator:
        y_pred = model(batch_x)

        loss = loss_fn(y_pred, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_iterator.set_description(f"Train loss: {loss.detach().cpu().numpy()}")


Network Initialized
Epoch:1


  0%|          | 0/66 [00:00<?, ?it/s]

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])


Train loss: 2.1950740814208984:   2%|▏         | 1/66 [00:02<03:11,  2.94s/it]

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])


...

Train loss: 2.06624436378479:  97%|█████████▋| 64/66 [00:39<00:01,  1.62it/s]  

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])


Train loss: 2.077425479888916:  98%|█████████▊| 65/66 [00:39<00:00,  1.56it/s]

Branch 1: torch.Size([32, 1, 170, 50])
Branch 2: torch.Size([32, 5, 170, 50])
Branch 3: torch.Size([32, 9, 170, 50])
Concat: torch.Size([32, 15, 170, 50])
Concat: torch.Size([32, 3, 170, 50])
Flatten: torch.Size([32, 25500])
Out: torch.Size([32, 9])


Train loss: 2.071622133255005: 100%|██████████| 66/66 [00:40<00:00,  1.64it/s]


In [83]:
model.eval()

all_inputs = []
all_outputs = []
all_labels = []

with torch.no_grad():
    test_iterator = tqdm(test_dataloader)
    for batch_x, batch_y, _, _ in test_iterator:
        out = model(batch_x)
        #print("True label:", batch_y, "Predicted label:", out)
        all_inputs.append(batch_x)
        all_outputs.append(out)
        all_labels.append(batch_y)

all_inputs  = torch.cat(all_inputs)
all_outputs = torch.cat(all_outputs)
all_labels  = torch.cat(all_labels)

test_loss = loss_fn(all_outputs, all_labels)
print(f"AVERAGE TEST LOSS: {test_loss}")


  0%|          | 0/17 [00:00<?, ?it/s]

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])


  6%|▌         | 1/17 [00:00<00:04,  3.86it/s]

Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])
Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])


 12%|█▏        | 2/17 [00:00<00:02,  5.26it/s]

...

Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])
Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])


 94%|█████████▍| 16/17 [00:02<00:00,  5.47it/s]

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])
Branch 1: torch.Size([24, 1, 170, 50])
Branch 2: torch.Size([24, 5, 170, 50])


100%|██████████| 17/17 [00:02<00:00,  5.73it/s]

Branch 3: torch.Size([24, 9, 170, 50])
Concat: torch.Size([24, 15, 170, 50])
Concat: torch.Size([24, 3, 170, 50])
Flatten: torch.Size([24, 25500])
Out: torch.Size([24, 9])
AVERAGE TEST LOSS: 2.0660295486450195


In [84]:
total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y, _, _ in test_dataloader:

        out = model(batch_x)

        loss = loss_fn(out, batch_y)
        total_loss += loss.item() * batch_x.size(0)

        pred = out.argmax(dim=1)

        correct += (pred == batch_y).sum().item()
        total += batch_y.size(0)

avg_loss = total_loss / total
accuracy = correct / total

print(f"Test loss: {avg_loss:.4f}")
print(f"Accuracy : {accuracy:.2%}")

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])
Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])
Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])
Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 9])


In [85]:
# Network definition with contrastive learning

NUM_LABELS = 9

class ContrastiveNet(nn.Module):
    def __init__(self):
        super().__init__()

        print("Network Initialized")

        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=5, kernel_size=2, stride=2),
            nn.ReLU())
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=3, kernel_size=1, stride=1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(in_channels=3, out_channels=6, kernel_size=2, stride=1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(in_channels=6, out_channels=9, kernel_size=4, stride=2, padding=1),
            nn.ReLU()
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=15, out_channels=3, kernel_size=1, stride=1, padding='same'),
            nn.ReLU()
        )

        #self.block2 = nn.Sequential(
            #nn.Dropout(0.2),
            #nn.LazyLinear(out_features=NUM_LABELS)
        #)

        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(25500, out_features=NUM_LABELS)
        )

        self.projection_head = nn.Sequential(
            nn.Linear(25500, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        print("Branch 1:", b1.shape)
        b2 = self.branch2(x)
        print("Branch 2:", b2.shape)
        b3 = self.branch3(x)
        print("Branch 3:", b3.shape)

        h = torch.cat([b1, b2, b3], dim=1)
        print("Concat:", h.shape)

        h = self.block1(h)
        print("Concat:", h.shape)
        h = h.flatten(1)
        print("Flatten:", h.shape)
        out = self.classifier(h) # classification
        print("Out:", out.shape)
        z = self.projection_head(h) # projection
        z = nn.functional.normalize(z, dim=1)

        return out,z

In [86]:
sample = train_dataset[0]
print(sample)

(tensor([[[0.0631, 0.0631, 0.0631,  ..., 0.0631, 0.0631, 0.0631],
         [0.0631, 0.0631, 0.0631,  ..., 0.0631, 0.0631, 0.0631],
         [0.0631, 0.0631, 0.0631,  ..., 0.0631, 0.0631, 0.0631],
         ...,
         [0.0631, 0.0631, 0.0631,  ..., 0.0631, 0.0631, 0.0631],
         [0.0631, 0.0631, 0.0631,  ..., 0.0631, 0.0631, 0.0631],
         [0.0631, 0.0631, 0.0631,  ..., 0.0631, 0.0631, 0.0631]]]), tensor(5), 0, 'S1a', tensor(3))


In [87]:
# CLASS PERSON DATASET

class PersonDataset(Dataset):
    def __init__(self, dataset_people):
        self.dataset_people = dataset_people

    def __len__(self):
        return len(self.dataset_people)

    def __getitem__(self, idx):
        sample = self.dataset_people[idx]

        # Convert to float32
        x = torch.from_numpy(sample["data"]).float()
        # Add channel dimension -> (1, 340, 100)
        x = x.unsqueeze(0)

        activity = torch.tensor(label_map[sample["label"]], dtype=torch.long)

        person = sample["person"]

        folder = sample["folder"]

        # stream = sample["stream"]

        return x, activity, person, folder

In [88]:
# GENERATION OF DATASET WRT PEOPLE

person_map = {
    "S1": 0,
    "S2": 0,
    "S3": 1,
    "S4": 0,
    "S5": 1,
    "S6": 0,
    "S7": 2
}

y_people = []

for folder in folders:

    set_name = folder[:2]
    person = person_map[set_name]

    y_people.append(person)

dataset_people = [
    {
        "data": data,
        "label": label,
        "person": person,
        "folder": folder
    }
    for data, label, person, folder
    in zip(X, y, y_people, folders)
]

In [89]:
print(len(dataset_people))

18680


In [90]:
# DATALOADER -> TRAIN E TEST DATA
labels = [
    sample["label"]
    for sample in dataset_people
]

train_data_people, test_data_people = train_test_split(
    dataset_people,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Campioni training:", len(train_data_people))
print("Campioni test:", len(test_data_people))

train_dataset_people = PersonDataset(train_data_people)
test_dataset_people = PersonDataset(test_data_people)

train_loader_people = DataLoader(
    train_dataset_people,
    batch_size=64,
    shuffle=True
)

test_loader_people = DataLoader(
    test_dataset_people,
    batch_size=64,
    shuffle=False
)

batch_x, batch_activity, batch_person, batch_folder = next(
    iter(train_loader_people)
)

print("Dati:", batch_x.shape)
print("Attività:", batch_activity.shape)
print("Persone:", batch_person.shape)
print("Cartelle:", len(batch_folder))

print("Persone presenti:", torch.unique(batch_person))

Campioni training: 14944
Campioni test: 3736
Dati: torch.Size([64, 1, 340, 100])
Attività: torch.Size([64])
Persone: torch.Size([64])
Cartelle: 64
Persone presenti: tensor([0, 1, 2])


In [91]:
# PER OGNI ATTIVITà -> CHE TIPO DI PERSONE

for activity in torch.unique(batch_activity):

    mask = batch_activity == activity
    people = torch.unique(batch_person[mask])

    print(
        f"Attività {activity.item()} "
        f"-> {mask.sum().item()} campioni "
        f"-> persone {people.tolist()}"
    )

Attività 0 -> 8 campioni -> persone [0, 1]
Attività 1 -> 9 campioni -> persone [0, 1, 2]
Attività 2 -> 9 campioni -> persone [0, 1]
Attività 3 -> 13 campioni -> persone [0, 1, 2]
Attività 4 -> 3 campioni -> persone [0, 1, 2]
Attività 5 -> 6 campioni -> persone [0, 1]
Attività 7 -> 9 campioni -> persone [0, 1, 2]
Attività 8 -> 7 campioni -> persone [0, 1, 2]


In [92]:
# LOSS FUNCTION NT-XENT-LOSS

def NT_Xent_loss(features, activities, people, temperature=0.5):

    #device = features.device
    batch_size = features.size(0)
    print(features.shape)

    # Similarity matrix
    similarity_score = torch.matmul(features, features.T) / temperature

    # POSITIVE KEYS -> DIFFERENT PEOPLE WITH SAME ACTIVITY
    same_activity = (activities.unsqueeze(dim=1) == activities.unsqueeze(dim=1))
    different_people = (people.unsqueeze(0) != people.unsqueeze(1))
    positive_keys = same_activity & different_people

    '''# True se due campioni hanno la stessa attività
    same_label = labels[:, None] == labels[None, :]

    # True se appartengono a soggetti diversi
    different_subject = subjects[:, None] != subjects[None, :]

    # Esclude il confronto di un elemento con se stesso
    self_mask = torch.eye(
        batch_size,
        dtype=torch.bool,
        device=features.device
    )

    # Positivi: stessa attività e soggetti diversi
    positive_mask = (
        same_label
        & different_subject
        & ~self_mask
    )

    # Negativi: attività diverse
    negative_mask = (
        ~same_label
        & ~self_mask
    )

    # Confronti utilizzati nella loss
    comparison_mask = positive_mask | negative_mask

    # Numero di positivi per ogni campione anchor
    positives_per_sample = positive_mask.sum(dim=1)

    # Usiamo soltanto gli anchor che hanno almeno un positivo
    valid_samples = positives_per_sample > 0

    if not valid_samples.any():
        return torch.tensor(
            0.0,
            device=features.device,
            requires_grad=True
    )

    # Esclude dalla loss:
    # - se stesso
    # - stessa attività e stesso soggetto
    masked_similarity = similarity_matrix.masked_fill(
        ~comparison_mask,
        float("-inf")
    )

    # Denominatore della NT-Xent
    log_denominator = torch.logsumexp(
        masked_similarity,
        dim=1
    )

    # Log-probabilità di ogni coppia
    log_probabilities = (
        similarity_matrix
        - log_denominator.unsqueeze(1)
    )

    # Consideriamo soltanto le coppie positive
    positive_log_probabilities = (
        log_probabilities
        * positive_mask.float()
    ).sum(dim=1)

    # Media sui positivi di ogni anchor
    loss_per_sample = (
        -positive_log_probabilities
        / positives_per_sample.clamp(min=1)
    )

    # Media soltanto sugli anchor validi
    loss = loss_per_sample[valid_samples].mean()

    positive_pairs = positive_mask.sum().item()

    return loss, positive_pairs'''


In [93]:
'''model = ContrastiveNet()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = opt.Adam(model.parameters(), lr=0.0001)

criterion = nn.CrossEntropyLoss()

epochs = 1

for epoch in range(epochs):

    model.train()
    print(f"Epoch: {epoch+1}")
    train_iterator = tqdm(train_loader)

    for x, activity, person, folder, stream in train_iterator:

        x = x.to(device)
        activity = activity.to(device)
        person = person.to(device)

        out, features = model(x)

        loss_ce = criterion(out, activity)

        loss_contrastive = NT_Xent_loss(
            features,
            activity,
            person
        )

        loss = loss_ce + 0.1 * loss_contrastive

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_iterator.set_description(
            f"CE: {loss_ce.item():.4f} | CTR: {loss_contrastive.item():.4f}"
        )

#batch_x, batch_y, _, _ = next(iter(train_dataloader))
#batch_x = batch_x.to(device)

#with torch.no_grad():
    #out, z = model(batch_x)

#print("Classificazione:", out.shape)
#print("Proiezione:", z.shape)'''

'model = ContrastiveNet()\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nmodel.to(device)\n\noptimizer = opt.Adam(model.parameters(), lr=0.0001)\n\ncriterion = nn.CrossEntropyLoss()\n\nepochs = 1\n\nfor epoch in range(epochs):\n\n    model.train()\n    print(f"Epoch: {epoch+1}")\n    train_iterator = tqdm(train_loader)\n\n    for x, activity, person, folder, stream in train_iterator:\n\n        x = x.to(device)\n        activity = activity.to(device)\n        person = person.to(device)\n\n        out, features = model(x)\n\n        loss_ce = criterion(out, activity)\n\n        loss_contrastive = NT_Xent_loss(\n            features,\n            activity,\n            person\n        )\n\n        loss = loss_ce + 0.1 * loss_contrastive\n\n        optimizer.zero_grad()\n        loss.backward()\n        optimizer.step()\n\n        train_iterator.set_description(\n            f"CE: {loss_ce.item():.4f} | CTR: {loss_contrastive.item():.4f}"\n        )\n\n#batch_x, batch_y, _, _ =

In [94]:
'''for x, activity, person, folder, stream in train_loader:

    out, features = model(x)

    loss_ce = criterion(out, activity)

    loss_contrastive = NT_Xent_loss(
        features,
        activity,
        person
    )'''

'for x, activity, person, folder, stream in train_loader:\n\n    out, features = model(x)\n\n    loss_ce = criterion(out, activity)\n\n    loss_contrastive = NT_Xent_loss(\n        features,\n        activity,\n        person\n    )'